# Criação estruturada de QA para avaliação de RAG

Este notebook monta uma base com 9 exemplos para avaliação:
- 3 perguntas `single-hop`
- 3 perguntas `multi-hop`
- 3 perguntas `globais`

Cada linha contém:
- pergunta
- trechos retirados dos documentos (`trecho_1`, `trecho_2`)
- resposta esperada
- metadados de fonte

Ao final, o dataset é salvo em CSV.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

In [2]:
# Resolve automaticamente a raiz do projeto (rodando da raiz ou de notebooks/)
_here = Path.cwd()
if (_here / "pdfs_txt").is_dir():
    PROJECT_ROOT = _here.resolve()
elif (_here.parent / "pdfs_txt").is_dir():
    PROJECT_ROOT = _here.parent.resolve()
else:
    raise FileNotFoundError(f"Não encontrei a pasta 'pdfs_txt' no cwd atual: {_here}")

DOCS_DIR = (PROJECT_ROOT / "pdfs_txt").resolve()
OUTPUT_DIR = (PROJECT_ROOT / "docs").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

txt_files = sorted(DOCS_DIR.glob("*.txt"))
if not txt_files:
    raise FileNotFoundError(f"Nenhum .txt encontrado em: {DOCS_DIR}")

docs = {p.name: p.read_text(encoding="utf-8", errors="ignore") for p in txt_files}
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Arquivos carregados:", len(docs))
print("Exemplos:", ", ".join(list(docs.keys())[:3]))

PROJECT_ROOT: /home/micael/mestrado/benchmarking-graphrag
Arquivos carregados: 1
Exemplos: Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt


In [10]:
# Gabarito curado manualmente a partir dos .txt em pdfs_txt/ (sem busca automática).


In [3]:
qa_rows: list[dict] = [
  {
    "id": "Q01",
    "tipo_pergunta": "single-hop",
    "pergunta": "Qual é a data do incidente na explosão da Accurate Energetic Systems?",
    "trecho_1": "Fatal Explosion at Accurate Energetic Systems McEwen, Tennessee | Incident Date: October 10, 2025",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "",
    "fonte_trecho_2": "",
    "resposta_esperada": "October 10, 2025.",
    "observacoes": ""
  },
  {
    "id": "Q02",
    "tipo_pergunta": "single-hop",
    "pergunta": "Quantos funcionários morreram no incidente descrito?",
    "trecho_1": "multiple catastrophic explosions fatally injured 16 employees at the Accurate Energetic Systems, LLC (“AES”) facility",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "",
    "fonte_trecho_2": "",
    "resposta_esperada": "16 funcionários.",
    "observacoes": ""
  },
  {
    "id": "Q03",
    "tipo_pergunta": "single-hop",
    "pergunta": "Qual era o nome do edifício onde ocorreu o incidente?",
    "trecho_1": "The incident occurred in Building 602, where, on the day of the incident, AES was manufacturing explosive charges",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "",
    "fonte_trecho_2": "",
    "resposta_esperada": "Building 602.",
    "observacoes": ""
  },
  {
    "id": "Q04",
    "tipo_pergunta": "multi-hop",
    "pergunta": "Quantos quilos de explosivos estavam presentes no prédio no momento do incidente e quantos foram consumidos nas detonações?",
    "trecho_1": "At the time of the incident, approximately 24,600 pounds of explosives were in the building",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "approximately 23,000 pounds of which were consumed in multiple detonations",
    "fonte_trecho_2": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "resposta_esperada": "Aproximadamente 24.600 libras estavam presentes e cerca de 23.000 libras foram consumidas.",
    "observacoes": "Combina duas informações numéricas do mesmo parágrafo."
  },
  {
    "id": "Q05",
    "tipo_pergunta": "multi-hop",
    "pergunta": "Quais materiais explosivos foram usados na composição do Comp B e qual sua proporção?",
    "trecho_1": "RDX Composition B (“Comp B”) – Comp B is a mixture of 60 percent RDX and 40 percent TNT",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "prepared by adding powdered or crystalline RDX to molten TNT",
    "fonte_trecho_2": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "resposta_esperada": "Comp B é composto por 60% RDX e 40% TNT.",
    "observacoes": "Exige identificar composição e materiais."
  },
  {
    "id": "Q06",
    "tipo_pergunta": "multi-hop",
    "pergunta": "Quais eram os horários de turno dos operadores e o que ocorreu às 7:47 a.m.?",
    "trecho_1": "AES scheduled first-shift operators to work 7:00 a.m. to 3:30 p.m., second-shift operators 3:00 p.m. to 11:30 p.m., and third shift operators 11:00 p.m. to 7:30 a.m.",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "At 7:47 a.m., the first explosion occurred in Building 602.",
    "fonte_trecho_2": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "resposta_esperada": "Turnos: 7:00–15:30, 15:00–23:30 e 23:00–7:30. Às 7:47 a.m. ocorreu a primeira explosão.",
    "observacoes": "Combina cronograma e evento."
  },
  {
    "id": "Q07",
    "tipo_pergunta": "global",
    "pergunta": "Qual organização conduziu a investigação descrita no documento?",
    "trecho_1": "U.S. Chemical Safety and Hazard Investigation Board",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "This document provides an update on the CSB’s investigation",
    "fonte_trecho_2": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "resposta_esperada": "U.S. Chemical Safety and Hazard Investigation Board (CSB).",
    "observacoes": "Síntese do documento."
  },
  {
    "id": "Q08",
    "tipo_pergunta": "global",
    "pergunta": "Quais foram alguns dos principais danos materiais causados pelo incidente?",
    "trecho_1": "Building 602 was completely destroyed as a result of the incident",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "nine additional buildings within the AES complex were damaged",
    "fonte_trecho_2": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "resposta_esperada": "O Building 602 foi completamente destruído e nove outros edifícios foram danificados.",
    "observacoes": "Requer visão geral do impacto."
  },
  {
    "id": "Q09",
    "tipo_pergunta": "global",
    "pergunta": "Quais são algumas das áreas principais que a investigação ainda está analisando?",
    "trecho_1": "The CSB is continuing to gather facts and analyze several key areas, including: Cause or probable cause... equipment design... sensitivities... industry guidance",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "",
    "fonte_trecho_2": "",
    "resposta_esperada": "Causas do evento, segurança de processos, design dos equipamentos, sensibilidade dos materiais e diretrizes da indústria.",
    "observacoes": ""
  },
  {
    "id": "Q10",
    "tipo_pergunta": "tabela",
    "pergunta": "Qual é a temperatura de decomposição do TNT sólido segundo a Tabela 1?",
    "trecho_1": "TNT(Solid)... Decomposition Temperature (℃)... 240",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "",
    "fonte_trecho_2": "",
    "resposta_esperada": "240 ℃.",
    "observacoes": "Leitura direta da tabela."
  },
  {
    "id": "Q11",
    "tipo_pergunta": "tabela",
    "pergunta": "Qual é a energia de impacto (J) associada ao PETN sólido na Tabela 1?",
    "trecho_1": "PETN(Solid)... Impact Energy (J)... 3.3",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "",
    "fonte_trecho_2": "",
    "resposta_esperada": "3.3 J.",
    "observacoes": ""
  },
  {
    "id": "Q12",
    "tipo_pergunta": "tabela",
    "pergunta": "Compare as alturas de queda (H50) para TNT sólido e Comp B sólido na Tabela 1.",
    "trecho_1": "TNT(Solid)... 0.95 m",
    "fonte_trecho_1": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "trecho_2": "Comp B(Solid)... 0.75 m",
    "fonte_trecho_2": "Accurate_Energetic_Solutions_Investigation_Update_Publication1.pdf",
    "resposta_esperada": "TNT sólido: 0.95 m; Comp B sólido: 0.75 m.",
    "observacoes": "Comparação entre duas entradas da tabela."
  }
]

qa_df = pd.DataFrame(qa_rows)
qa_df

,id,tipo_pergunta,pergunta,trecho_1,fonte_trecho_1,trecho_2,fonte_trecho_2,resposta_esperada,observacoes
0,Q01,single-hop,Qual é a data do incidente na explosão da Accu...,Fatal Explosion at Accurate Energetic Systems ...,Accurate_Energetic_Solutions_Investigation_Upd...,,,"October 10, 2025.",
1,Q02,single-hop,Quantos funcionários morreram no incidente des...,multiple catastrophic explosions fatally injur...,Accurate_Energetic_Solutions_Investigation_Upd...,,,16 funcionários.,
2,Q03,single-hop,Qual era o nome do edifício onde ocorreu o inc...,"The incident occurred in Building 602, where, ...",Accurate_Energetic_Solutions_Investigation_Upd...,,,Building 602.,
3,Q04,multi-hop,Quantos quilos de explosivos estavam presentes...,"At the time of the incident, approximately 24,...",Accurate_Energetic_Solutions_Investigation_Upd...,"approximately 23,000 pounds of which were cons...",Accurate_Energetic_Solutions_Investigation_Upd...,Aproximadamente 24.600 libras estavam presente...,Combina duas informações numéricas do mesmo pa...
4,Q05,multi-hop,Quais materiais explosivos foram usados na com...,RDX Composition B (“Comp B”) – Comp B is a mix...,Accurate_Energetic_Solutions_Investigation_Upd...,prepared by adding powdered or crystalline RDX...,Accurate_Energetic_Solutions_Investigation_Upd...,Comp B é composto por 60% RDX e 40% TNT.,Exige identificar composição e materiais.
5,Q06,multi-hop,Quais eram os horários de turno dos operadores...,AES scheduled first-shift operators to work 7:...,Accurate_Energetic_Solutions_Investigation_Upd...,"At 7:47 a.m., the first explosion occurred in ...",Accurate_Energetic_Solutions_Investigation_Upd...,"Turnos: 7:00–15:30, 15:00–23:30 e 23:00–7:30. ...",Combina cronograma e evento.
6,Q07,global,Qual organização conduziu a investigação descr...,U.S. Chemical Safety and Hazard Investigation ...,Accurate_Energetic_Solutions_Investigation_Upd...,This document provides an update on the CSB’s ...,Accurate_Energetic_Solutions_Investigation_Upd...,U.S. Chemical Safety and Hazard Investigation ...,Síntese do documento.
7,Q08,global,Quais foram alguns dos principais danos materi...,Building 602 was completely destroyed as a res...,Accurate_Energetic_Solutions_Investigation_Upd...,nine additional buildings within the AES compl...,Accurate_Energetic_Solutions_Investigation_Upd...,O Building 602 foi completamente destruído e n...,Requer visão geral do impacto.
8,Q09,global,Quais são algumas das áreas principais que a i...,The CSB is continuing to gather facts and anal...,Accurate_Energetic_Solutions_Investigation_Upd...,,,"Causas do evento, segurança de processos, desi...",
9,Q10,tabela,Qual é a temperatura de decomposição do TNT só...,TNT(Solid)... Decomposition Temperature (℃)......,Accurate_Energetic_Solutions_Investigation_Upd...,,,240 ℃.,Leitura direta da tabela.


In [4]:
# Edite as colunas 'pergunta' e 'resposta_esperada' e, se necessário, ajuste os trechos.
# Validação rápida das quantidades por tipo:
counts = qa_df["tipo_pergunta"].value_counts().to_dict()
expected = {"single-hop": 3, "multi-hop": 3, "global": 3}
assert counts == expected, f"Contagem inválida por tipo. Esperado {expected}, obtido {counts}"

# Garante que não restaram placeholders antes de exportar.
placeholder_pattern = r"\[TODO\]"
placeholder_mask = qa_df["pergunta"].str.contains(placeholder_pattern, regex=True) | qa_df["resposta_esperada"].str.contains(placeholder_pattern, regex=True)
if placeholder_mask.any():
    print("Ainda existem placeholders [TODO]. Edite antes de exportar.")
    display(qa_df.loc[placeholder_mask, ["id", "tipo_pergunta", "pergunta", "resposta_esperada"]])
else:
    print("Tudo certo para exportar.")

AssertionError: Contagem inválida por tipo. Esperado {'single-hop': 3, 'multi-hop': 3, 'global': 3}, obtido {'single-hop': 3, 'multi-hop': 3, 'global': 3, 'tabela': 3}

In [5]:
OUTPUT_CSV = OUTPUT_DIR / "rag_eval_qa_estruturado.csv"
qa_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"CSV salvo em: {OUTPUT_CSV}")
print(f"Total de linhas: {len(qa_df)}")

CSV salvo em: /home/micael/mestrado/benchmarking-graphrag/docs/rag_eval_qa_estruturado.csv
Total de linhas: 12
